In [ ]:
# prepare_all_subjects_64hz.py

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt



# ============================================================
# Config
# ============================================================

PROJECT_ROOT = Path("/home/iailab42/khans1/projects")

ROOT_DIR = Path("/home/iailab42/g2-synthetic/PPG_FieldStudy")

DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "figures" / "preprocessing"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = DATA_DIR

SUBJECT_IDS = list(range(1, 16))  # S1 to S15

TARGET_HZ = 64

WINDOW_LEN = 512       # 8 seconds * 64 Hz
SHIFT_LEN = 128        # 2 seconds * 64 Hz = 75% overlap
MIN_LABEL_COVERAGE = 0.80
DROP_ACTIVITY_ZERO = True

CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]

SAVE_RAW = True
SAVE_DEBUG_CSVS = True
NUM_DEBUG_WINDOWS = 10
SAVE_PLOTS = True
NUM_SIGNAL_EXAMPLES = 3
PSD_MAX_WINDOWS = 200


# ------------------------------------------------------------
# S6 special handling
# ------------------------------------------------------------
# PPG-DaLiA readme says S6 had a hardware issue and S6.pkl
# only contains the valid part, about the first 1.5 hours.
# This guard is added just in case your local S6.pkl is longer.
S6_MAX_VALID_SECONDS = 90 * 60  # 1.5 hours = 5400 seconds
SPECIAL_VALID_SECONDS = {
    "S6": S6_MAX_VALID_SECONDS,
}


# ============================================================
# Load pickle
# ============================================================

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")


# ============================================================
# Shape helpers
# ============================================================

def ensure_2d(x):
    """
    Ensure signal is [time, channels].
    BVP/EDA/TEMP may sometimes appear as [time], so convert to [time, 1].
    """
    x = np.asarray(x, dtype=np.float32)

    if x.ndim == 1:
        x = x[:, None]

    if x.ndim != 2:
        raise ValueError(f"Expected 1D or 2D signal, got shape {x.shape}")

    return x


# ============================================================
# Resampling helpers
# ============================================================

def resample_linear(x, source_hz, target_hz):
    """
    Linear interpolation for continuous signals:
    ACC, BVP, EDA, TEMP.

    Input:
        x: [time] or [time, channels]

    Output:
        resampled x as [time, channels]
    """
    x = ensure_2d(x)

    if source_hz == target_hz:
        return x.copy()

    old_len = x.shape[0]
    duration = old_len / float(source_hz)
    new_len = int(round(duration * target_hz))

    old_t = np.arange(old_len) / float(source_hz)
    new_t = np.arange(new_len) / float(target_hz)

    out = np.zeros((new_len, x.shape[1]), dtype=np.float32)

    for c in range(x.shape[1]):
        out[:, c] = np.interp(new_t, old_t, x[:, c])

    return out


def resample_labels_nearest(labels, source_hz, target_hz):
    """
    Nearest-neighbor resampling for activity labels.

    Important:
    Do NOT linearly interpolate activity labels.
    """
    labels = np.asarray(labels).reshape(-1).astype(np.int64)

    old_len = len(labels)
    duration = old_len / float(source_hz)
    new_len = int(round(duration * target_hz))

    new_t = np.arange(new_len) / float(target_hz)
    old_idx = np.round(new_t * source_hz).astype(np.int64)
    old_idx = np.clip(old_idx, 0, old_len - 1)

    return labels[old_idx]


# ============================================================
# Windowing
# ============================================================

def make_windows(
    signal,
    activity,
    window_len=512,
    shift_len=128,
    min_label_coverage=0.80,
    drop_activity_zero=True,
):
    """
    Create windows from one subject.

    signal:
        [time, 6]

    activity:
        [time]

    Output:
        X: [N, 512, 6]
        y: [N]
        coverage: [N]
        start_indices: [N]
        end_indices: [N]
    """
    X_list = []
    y_list = []
    coverage_list = []
    start_list = []
    end_list = []

    n = signal.shape[0]

    for start in range(0, n - window_len + 1, shift_len):
        end = start + window_len

        x_win = signal[start:end]
        a_win = activity[start:end]

        labels, counts = np.unique(a_win, return_counts=True)

        dominant_index = np.argmax(counts)
        dominant_label = int(labels[dominant_index])
        coverage = counts[dominant_index] / float(window_len)

        if drop_activity_zero and dominant_label == 0:
            continue

        if coverage < min_label_coverage:
            continue

        if not np.all(np.isfinite(x_win)):
            continue

        X_list.append(x_win.astype(np.float32))
        y_list.append(dominant_label)
        coverage_list.append(coverage)
        start_list.append(start)
        end_list.append(end)

    if len(X_list) == 0:
        return None, None, None, None, None

    X = np.stack(X_list).astype(np.float32)
    y = np.asarray(y_list, dtype=np.int64)
    coverage = np.asarray(coverage_list, dtype=np.float32)
    start_indices = np.asarray(start_list, dtype=np.int64)
    end_indices = np.asarray(end_list, dtype=np.int64)

    return X, y, coverage, start_indices, end_indices


# ============================================================
# Subject processing
# ============================================================

def process_subject(subject_id):
    subject_name = f"S{subject_id}"
    pkl_path = ROOT_DIR / subject_name / f"{subject_name}.pkl"

    if not pkl_path.exists():
        print(f"\nSkipping {subject_name}: file not found:")
        print(" ", pkl_path)
        return None

    print("\n" + "=" * 70)
    print(f"Processing {subject_name}")
    print("File:", pkl_path)

    data = load_pkl(pkl_path)

    wrist = data["signal"]["wrist"]

    # Original PPG-DaLiA wrist signals
    acc = ensure_2d(wrist["ACC"])       # 32 Hz, [T, 3]
    bvp = ensure_2d(wrist["BVP"])       # 64 Hz, [T, 1]
    eda = ensure_2d(wrist["EDA"])       # 4 Hz,  [T, 1]
    temp = ensure_2d(wrist["TEMP"])     # 4 Hz,  [T, 1]
    activity = np.asarray(data["activity"]).reshape(-1).astype(np.int64)  # 4 Hz labels

    print("Original shapes:")
    print("  ACC:     ", acc.shape)
    print("  BVP:     ", bvp.shape)
    print("  EDA:     ", eda.shape)
    print("  TEMP:    ", temp.shape)
    print("  activity:", activity.shape)

    # Convert all wrist signals to 64 Hz
    acc_64 = resample_linear(acc, source_hz=32, target_hz=TARGET_HZ)
    bvp_64 = resample_linear(bvp, source_hz=64, target_hz=TARGET_HZ)
    eda_64 = resample_linear(eda, source_hz=4, target_hz=TARGET_HZ)
    temp_64 = resample_linear(temp, source_hz=4, target_hz=TARGET_HZ)

    # Convert activity labels to 64 Hz using nearest-neighbor
    activity_64 = resample_labels_nearest(
        activity,
        source_hz=4,
        target_hz=TARGET_HZ,
    )

    # Crop all signals to same length
    min_len_before_s6_guard = min(
        len(acc_64),
        len(bvp_64),
        len(eda_64),
        len(temp_64),
        len(activity_64),
    )

    min_len = min_len_before_s6_guard
    s6_special_crop_applied = False

    # Explicit S6 guard
    if subject_name in SPECIAL_VALID_SECONDS:
        max_valid_seconds = SPECIAL_VALID_SECONDS[subject_name]
        max_valid_samples = int(round(max_valid_seconds * TARGET_HZ))

        print(f"\n{subject_name} special handling:")
        print(f"  Max valid duration allowed: {max_valid_seconds} sec")
        print(f"  Max valid samples at {TARGET_HZ} Hz: {max_valid_samples}")
        print(f"  Current aligned length before S6 guard: {min_len} samples")
        print(f"  Current aligned duration before S6 guard: {min_len / TARGET_HZ:.2f} sec")

        if min_len > max_valid_samples:
            min_len = max_valid_samples
            s6_special_crop_applied = True
            print(f"  Cropping {subject_name} to first {max_valid_seconds} seconds.")
        else:
            print(f"  No extra crop needed. {subject_name}.pkl already appears to contain only valid part.")

    acc_64 = acc_64[:min_len]
    bvp_64 = bvp_64[:min_len]
    eda_64 = eda_64[:min_len]
    temp_64 = temp_64[:min_len]
    activity_64 = activity_64[:min_len]

    # Combine into one [time, 6] matrix:
    # ACC_x, ACC_y, ACC_z, BVP, EDA, TEMP
    signal_64 = np.concatenate(
        [acc_64, bvp_64, eda_64, temp_64],
        axis=1,
    ).astype(np.float32)

    if signal_64.shape[1] != 6:
        raise ValueError(
            f"{subject_name}: expected 6 channels after concat, got {signal_64.shape}"
        )

    print("\nAfter resampling/cropping:")
    print("  signal_64:  ", signal_64.shape)
    print("  activity_64:", activity_64.shape)
    print("  duration:   ", f"{len(signal_64) / TARGET_HZ:.2f} sec")

    # Create 8-second windows with 2-second shift
    X, y, coverage, start_indices, end_indices = make_windows(
        signal=signal_64,
        activity=activity_64,
        window_len=WINDOW_LEN,
        shift_len=SHIFT_LEN,
        min_label_coverage=MIN_LABEL_COVERAGE,
        drop_activity_zero=DROP_ACTIVITY_ZERO,
    )

    if X is None:
        print(f"{subject_name}: no valid windows after filtering.")
        return None

    subjects = np.array([subject_name] * len(y), dtype=object)

    print("\nWindowed:")
    print("  X:", X.shape)
    print("  y:", y.shape)
    print("  label counts:", dict(zip(*np.unique(y, return_counts=True))))

    subject_summary = {
        "subject": subject_name,
        "min_len_before_s6_guard": int(min_len_before_s6_guard),
        "min_len_after_s6_guard": int(min_len),
        "duration_before_s6_guard_sec": float(min_len_before_s6_guard / TARGET_HZ),
        "duration_after_s6_guard_sec": float(min_len / TARGET_HZ),
        "s6_special_crop_applied": bool(s6_special_crop_applied),
        "num_windows_after_filtering": int(len(y)),
    }

    return {
        "X": X,
        "y": y,
        "subjects": subjects,
        "coverage": coverage,
        "start_indices": start_indices,
        "end_indices": end_indices,
        "summary": subject_summary,
    }



# ============================================================
# Plotting helpers
# ============================================================

def save_bar_plot(labels, counts, title, xlabel, ylabel, save_path, rotate=0):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([str(x) for x in labels], counts)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)


def plot_activity_distribution(y, out_dir):
    labels, counts = np.unique(y, return_counts=True)
    save_bar_plot(
        labels=labels,
        counts=counts,
        title="Activity Window Distribution",
        xlabel="Activity label",
        ylabel="Number of windows",
        save_path=out_dir / "activity_distribution.png",
    )


def plot_subject_distribution(subjects, out_dir):
    labels, counts = np.unique(subjects, return_counts=True)
    save_bar_plot(
        labels=labels,
        counts=counts,
        title="Subject Window Distribution",
        xlabel="Subject",
        ylabel="Number of windows",
        save_path=out_dir / "subject_distribution.png",
        rotate=45,
    )


def plot_coverage_distribution(coverage, out_dir):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(coverage, bins=30)
    ax.set_title("Dominant Activity Coverage Distribution")
    ax.set_xlabel("Dominant label coverage")
    ax.set_ylabel("Number of windows")
    fig.tight_layout()
    fig.savefig(out_dir / "coverage_distribution.png", dpi=150)
    plt.close(fig)


def plot_channel_stats(X_raw, X_norm, out_dir):
    raw_mean = X_raw.mean(axis=(0, 1))
    raw_std = X_raw.std(axis=(0, 1))
    norm_mean = X_norm.mean(axis=(0, 1))
    norm_std = X_norm.std(axis=(0, 1))

    x = np.arange(len(CHANNEL_NAMES))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, raw_mean, width, label="Raw mean")
    ax.bar(x + width / 2, raw_std, width, label="Raw std")
    ax.set_title("Raw Channel Statistics")
    ax.set_xticks(x)
    ax.set_xticklabels(CHANNEL_NAMES, rotation=30)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "raw_channel_statistics.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, norm_mean, width, label="Normalized mean")
    ax.bar(x + width / 2, norm_std, width, label="Normalized std")
    ax.set_title("Normalized Channel Statistics")
    ax.set_xticks(x)
    ax.set_xticklabels(CHANNEL_NAMES, rotation=30)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "normalized_channel_statistics.png", dpi=150)
    plt.close(fig)


def plot_signal_examples(X_raw, X_norm, y, subjects, out_dir, num_examples=3):
    n = min(num_examples, len(y))
    time_sec = np.arange(X_raw.shape[1]) / float(TARGET_HZ)

    for i in range(n):
        fig, axes = plt.subplots(len(CHANNEL_NAMES), 1, figsize=(12, 10), sharex=True)
        for c, ax in enumerate(axes):
            ax.plot(time_sec, X_raw[i, :, c])
            ax.set_ylabel(CHANNEL_NAMES[c])
        axes[0].set_title(f"Raw window example {i} | subject={subjects[i]} | label={y[i]}")
        axes[-1].set_xlabel("Time (seconds)")
        fig.tight_layout()
        fig.savefig(out_dir / f"raw_window_example_{i:02d}.png", dpi=150)
        plt.close(fig)

        fig, axes = plt.subplots(len(CHANNEL_NAMES), 1, figsize=(12, 10), sharex=True)
        for c, ax in enumerate(axes):
            ax.plot(time_sec, X_norm[i, :, c])
            ax.set_ylabel(CHANNEL_NAMES[c])
        axes[0].set_title(f"Normalized window example {i} | subject={subjects[i]} | label={y[i]}")
        axes[-1].set_xlabel("Time (seconds)")
        fig.tight_layout()
        fig.savefig(out_dir / f"normalized_window_example_{i:02d}.png", dpi=150)
        plt.close(fig)


def plot_correlation_heatmap(X_norm, out_dir, max_windows=2000):
    n = min(max_windows, X_norm.shape[0])
    flat = X_norm[:n].reshape(-1, X_norm.shape[-1])
    corr = np.corrcoef(flat, rowvar=False)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, vmin=-1, vmax=1)
    ax.set_title("Channel Correlation Heatmap")
    ax.set_xticks(np.arange(len(CHANNEL_NAMES)))
    ax.set_yticks(np.arange(len(CHANNEL_NAMES)))
    ax.set_xticklabels(CHANNEL_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CHANNEL_NAMES)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_dir / "channel_correlation_heatmap.png", dpi=150)
    plt.close(fig)

    np.save(out_dir / "channel_correlation_matrix.npy", corr.astype(np.float32))


def plot_psd_examples(X_norm, out_dir, max_windows=200):
    n = min(max_windows, X_norm.shape[0])
    x = X_norm[:n]
    freqs = np.fft.rfftfreq(X_norm.shape[1], d=1.0 / TARGET_HZ)

    fig, ax = plt.subplots(figsize=(10, 6))
    for c, name in enumerate(CHANNEL_NAMES):
        fft_vals = np.fft.rfft(x[:, :, c], axis=1)
        psd = (np.abs(fft_vals) ** 2).mean(axis=0)
        ax.plot(freqs, psd, label=name)

    ax.set_title("Average Power Spectral Density")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power")
    ax.set_xlim(0, TARGET_HZ / 2)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "average_psd.png", dpi=150)
    plt.close(fig)


def save_preprocessing_plots(X_raw, X_norm, y, subjects, coverage, out_dir):
    plot_dir = out_dir / "plots" / "preprocessing"
    plot_dir.mkdir(parents=True, exist_ok=True)

    plot_activity_distribution(y, plot_dir)
    plot_subject_distribution(subjects, plot_dir)
    plot_coverage_distribution(coverage, plot_dir)
    plot_channel_stats(X_raw, X_norm, plot_dir)
    plot_signal_examples(X_raw, X_norm, y, subjects, plot_dir, NUM_SIGNAL_EXAMPLES)
    plot_correlation_heatmap(X_norm, plot_dir)
    plot_psd_examples(X_norm, plot_dir, PSD_MAX_WINDOWS)

    print("Saved preprocessing plots to:", plot_dir)


# ============================================================
# Debug CSVs
# ============================================================

def save_debug_csvs(out_dir, X_norm, y, subjects, coverage, start_indices, num_debug_windows=10):
    debug_dir = out_dir / "debug_csvs"
    debug_dir.mkdir(parents=True, exist_ok=True)

    n_debug = min(num_debug_windows, len(y))

    for i in range(n_debug):
        df = pd.DataFrame(X_norm[i], columns=CHANNEL_NAMES)

        df.insert(0, "time_step", np.arange(X_norm.shape[1]))
        df.insert(1, "time_sec", np.arange(X_norm.shape[1]) / float(TARGET_HZ))
        df["activity_label"] = int(y[i])
        df["subject"] = str(subjects[i])
        df["dominant_label_coverage"] = float(coverage[i])
        df["start_sample_64hz"] = int(start_indices[i])

        csv_name = f"window_{i:04d}_{subjects[i]}_label_{y[i]}.csv"
        df.to_csv(debug_dir / csv_name, index=False)

    print(f"Saved {n_debug} debug CSV windows to:", debug_dir)


# ============================================================
# Main
# ============================================================

def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("PPG-DaLiA all-subject preprocessing")
    print("Root dir:", ROOT_DIR)
    print("Output dir:", OUT_DIR)
    print("Target Hz:", TARGET_HZ)
    print("Window len:", WINDOW_LEN, "samples =", WINDOW_LEN / TARGET_HZ, "seconds")
    print("Shift len:", SHIFT_LEN, "samples =", SHIFT_LEN / TARGET_HZ, "seconds")

    overlap_samples = WINDOW_LEN - SHIFT_LEN
    overlap_seconds = overlap_samples / TARGET_HZ
    overlap_percent = overlap_samples / WINDOW_LEN * 100.0

    print(
        f"Overlap: {overlap_samples} samples = "
        f"{overlap_seconds:.2f} seconds = {overlap_percent:.1f}%"
    )

    X_all_list = []
    y_all_list = []
    subject_all_list = []
    coverage_all_list = []
    start_all_list = []
    end_all_list = []
    summary_list = []

    for subject_id in SUBJECT_IDS:
        result = process_subject(subject_id)

        if result is None:
            continue

        X_all_list.append(result["X"])
        y_all_list.append(result["y"])
        subject_all_list.append(result["subjects"])
        coverage_all_list.append(result["coverage"])
        start_all_list.append(result["start_indices"])
        end_all_list.append(result["end_indices"])
        summary_list.append(result["summary"])

    if len(X_all_list) == 0:
        raise RuntimeError(
            "No subjects were processed. Check ROOT_DIR and folder structure."
        )

    # Concatenate subjects in S1, S2, ..., S15 order.
    # Within each subject, windows remain chronological.
    X_all_raw = np.concatenate(X_all_list, axis=0).astype(np.float32)
    y_all = np.concatenate(y_all_list, axis=0).astype(np.int64)
    subjects_all = np.concatenate(subject_all_list, axis=0)
    coverage_all = np.concatenate(coverage_all_list, axis=0).astype(np.float32)
    start_indices_all = np.concatenate(start_all_list, axis=0).astype(np.int64)
    end_indices_all = np.concatenate(end_all_list, axis=0).astype(np.int64)

    print("\n" + "=" * 70)
    print("Combined dataset before normalization")
    print("X_all_raw:", X_all_raw.shape)
    print("y_all:", y_all.shape)
    print("subjects_all:", subjects_all.shape)
    print("coverage_all:", coverage_all.shape)

    print("\nOverall label counts:")
    print(dict(zip(*np.unique(y_all, return_counts=True))))

    print("\nSubject window counts:")
    print(dict(zip(*np.unique(subjects_all, return_counts=True))))

    # Global normalization across all processed windows.
    # For strict subject-independent downstream evaluation, compute train-subject
    # mean/std separately later. For diffusion training on all subjects, this is okay.
    mean = X_all_raw.mean(axis=(0, 1), keepdims=True)
    std = X_all_raw.std(axis=(0, 1), keepdims=True)
    std = np.maximum(std, 1e-8)

    X_all_norm = ((X_all_raw - mean) / std).astype(np.float32)

    print("\nGlobal normalization stats:")
    for name, m, s in zip(CHANNEL_NAMES, mean.reshape(-1), std.reshape(-1)):
        print(f"  {name:6s}: mean={m: .6f}, std={s: .6f}")

    print("\nCheck after normalization:")
    for name, m, s in zip(
        CHANNEL_NAMES,
        X_all_norm.mean(axis=(0, 1)),
        X_all_norm.std(axis=(0, 1)),
    ):
        print(f"  {name:6s}: mean={m: .6f}, std={s: .6f}")

    # Save full 6-channel dataset
    np.save(OUT_DIR / "all_X.npy", X_all_norm)
    np.save(OUT_DIR / "all_y.npy", y_all)
    np.save(OUT_DIR / "all_subject.npy", subjects_all)
    np.save(OUT_DIR / "all_coverage.npy", coverage_all)
    np.save(OUT_DIR / "all_start_indices_64hz.npy", start_indices_all)
    np.save(OUT_DIR / "all_end_indices_64hz.npy", end_indices_all)

    # Save ACC+BVP only version for your current diffusion code.
    # Channels: 0,1,2,3 = ACC_x, ACC_y, ACC_z, BVP
    X_all_acc_bvp = X_all_norm[:, :, [0, 1, 2, 3]].astype(np.float32)
    np.save(OUT_DIR / "all_X_acc_bvp.npy", X_all_acc_bvp)

    # Save raw if requested
    if SAVE_RAW:
        np.save(OUT_DIR / "all_X_raw.npy", X_all_raw)
        X_all_raw_acc_bvp = X_all_raw[:, :, [0, 1, 2, 3]].astype(np.float32)
        np.save(OUT_DIR / "all_X_raw_acc_bvp.npy", X_all_raw_acc_bvp)

    # Save normalization stats
    np.savez(
        OUT_DIR / "normalization_stats.npz",
        mean=mean.astype(np.float32),
        std=std.astype(np.float32),
        channel_names=np.array(CHANNEL_NAMES),
        target_hz=np.array([TARGET_HZ], dtype=np.int64),
        window_len=np.array([WINDOW_LEN], dtype=np.int64),
        shift_len=np.array([SHIFT_LEN], dtype=np.int64),
        min_label_coverage=np.array([MIN_LABEL_COVERAGE], dtype=np.float32),
        drop_activity_zero=np.array([DROP_ACTIVITY_ZERO]),
        s6_max_valid_seconds=np.array([S6_MAX_VALID_SECONDS], dtype=np.int64),
    )

    # Save metadata CSV
    metadata_df = pd.DataFrame({
        "global_window_id": np.arange(len(y_all)),
        "subject": subjects_all,
        "activity_label": y_all,
        "dominant_label_coverage": coverage_all,
        "start_sample_64hz": start_indices_all,
        "end_sample_64hz": end_indices_all,
        "start_time_sec": start_indices_all / float(TARGET_HZ),
        "end_time_sec": end_indices_all / float(TARGET_HZ),
    })

    metadata_df.to_csv(OUT_DIR / "all_metadata.csv", index=False)

    # Save subject summary CSV, including S6 special handling info
    subject_summary_df = pd.DataFrame(summary_list)
    subject_summary_df.to_csv(OUT_DIR / "subject_summary.csv", index=False)

    if SAVE_PLOTS:
        save_preprocessing_plots(
            X_raw=X_all_raw,
            X_norm=X_all_norm,
            y=y_all,
            subjects=subjects_all,
            coverage=coverage_all,
            out_dir=FIGURE_DIR,
        )

    if SAVE_DEBUG_CSVS:
        save_debug_csvs(
            out_dir=OUT_DIR,
            X_norm=X_all_norm,
            y=y_all,
            subjects=subjects_all,
            coverage=coverage_all,
            start_indices=start_indices_all,
            num_debug_windows=NUM_DEBUG_WINDOWS,
        )

    print("\n" + "=" * 70)
    print("Saved files to:", OUT_DIR)

    print("\nMain files:")
    print(" ", OUT_DIR / "all_X.npy", X_all_norm.shape, "[N, 512, 6]")
    print(" ", OUT_DIR / "all_y.npy", y_all.shape)
    print(" ", OUT_DIR / "all_subject.npy", subjects_all.shape)
    print(" ", OUT_DIR / "all_X_acc_bvp.npy", X_all_acc_bvp.shape, "[N, 512, 4]")

    if SAVE_RAW:
        print("\nRaw files:")
        print(" ", OUT_DIR / "all_X_raw.npy", X_all_raw.shape)
        print(" ", OUT_DIR / "all_X_raw_acc_bvp.npy", X_all_raw_acc_bvp.shape)

    print("\nMetadata:")
    print(" ", OUT_DIR / "all_metadata.csv")
    print(" ", OUT_DIR / "subject_summary.csv")
    print(" ", OUT_DIR / "normalization_stats.npz")

    print("\nCheck S6 in subject_summary.csv:")
    print("  If s6_special_crop_applied=False, then S6.pkl was already short/valid.")
    print("  If s6_special_crop_applied=True, then this script cropped S6 to first 90 minutes.")

    print("\nDone.")


if __name__ == "__main__":
    main()

PPG-DaLiA all-subject preprocessing
Root dir: /home/iailab42/g2-synthetic/PPG_FieldStudy
Output dir: /home/iailab42/khans1/projects/data
Target Hz: 64
Window len: 512 samples = 8.0 seconds
Shift len: 128 samples = 2.0 seconds
Overlap: 384 samples = 6.00 seconds = 75.0%

Processing S1
File: /home/iailab42/g2-synthetic/PPG_FieldStudy/S1/S1.pkl
Original shapes:
  ACC:      (294784, 3)
  BVP:      (589568, 1)
  EDA:      (36848, 1)
  TEMP:     (36848, 1)
  activity: (36848,)

After resampling/cropping:
  signal_64:   (589568, 6)
  activity_64: (589568,)
  duration:    9212.00 sec

Windowed:
  X: (3445, 512, 6)
  y: (3445,)
  label counts: {np.int64(1): np.int64(347), np.int64(2): np.int64(141), np.int64(3): np.int64(170), np.int64(4): np.int64(203), np.int64(5): np.int64(442), np.int64(6): np.int64(1175), np.int64(7): np.int64(375), np.int64(8): np.int64(592)}

Processing S2
File: /home/iailab42/g2-synthetic/PPG_FieldStudy/S2/S2.pkl
Original shapes:
  ACC:      (262560, 3)
  BVP:      (525